In [1]:
#--------------------------------------------------------------------------------------------------------------
#
#  Code: AA_003_run_RAG_test_without_using_AutoGen_20260422.ipynb
#
#  Goal: To write some Python code (without using AutoGen) to implement Standard (Linear) Retrieval-Augmented Generation (RAG)
#        And read one CCAR review methodology PDF file, and then draw top 3 summary sentences / pargraphs
#  
#        Jingru Chen
#
#  Date:  2026-04-22
#
#---------------------------------------------------------------------------------------------------------------

In [2]:
from datetime import datetime
from zoneinfo import ZoneInfo

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM ED

2026-04-23 00:36:55 EDT
2026-04-23 12:36:55 AM EDT


In [3]:
from sentence_transformers import SentenceTransformer

# Current: 384 dimensions (Fast, but lower capacity)
# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Upgrade: 768 dimensions (Higher accuracy, captures more nuance)
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# Extreme Upgrade: 1024+ dimensions (e.g., using BGE or Instructor models)
# embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
pwd

'C:\\Users\\chen_\\gemini_study'

In [5]:
ls 

 Volume in drive C is Windows-SSD
 Volume Serial Number is BE1E-EB95

 Directory of C:\Users\chen_\gemini_study

04/23/2026  12:36 AM    <DIR>          .
04/22/2026  09:40 PM    <DIR>          ..
04/22/2026  10:19 PM                54 .env
04/22/2026  11:02 PM    <DIR>          .ipynb_checkpoints
04/22/2026  09:09 PM                 5 .python-version
04/22/2026  09:15 PM    <DIR>          .venv
04/23/2026  12:36 AM           127,305 AA_run_RAG_test_without_using_AutoGen_20260422.ipynb
03/12/2026  11:59 AM         1,640,567 CCAR_Review_Methodology_2012.pdf
03/12/2026  12:12 PM         1,172,154 CECL_implementation_model_risk_WP_2024_03_version.pdf
04/23/2026  12:36 AM            58,708 chen_RAG_test_20260422.ipynb
04/22/2026  09:29 PM             4,600 doc.md
04/22/2026  09:09 PM                90 main.py
04/22/2026  10:02 PM             5,503 NASA_moom_mission_20240416.md
04/22/2026  09:14 PM               274 pyproject.toml
04/22/2026  09:09 PM                 0 README.md
04/22/2026  

In [6]:
%pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


C:\Users\chen_\AppData\Local\uv\cache\builds-v0\.tmpnO11e9\Scripts\python.exe: No module named pip


In [7]:
### I start Jupyter notebook via two following commands. thus I need to use ( !uv pip install pymupdf  ).

#   uv add sentence_transformers chromadb google-genai python-dotenv
#   uv run --with jupyter jupyter lab

In [8]:
!uv pip install pymupdf   

Checked 1 package in 0.97ms


In [9]:
from sentence_transformers import SentenceTransformer

# (1) An all-rounder (balancing speed and accuracy)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# (2) Choose a more powerful model (larger footprint, but higher accuracy).
# embedding_model = SentenceTransformer("all-mpnet-base-v2")

# If my use case is RAG or involves processing long-form text
# embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Step 1: Upload one CECL document PDF file for RAG summarization

In [10]:
from typing import List

import sys
!{sys.executable} -m pip install pymupdf

import fitz  # after-uploading name of PyMuPDF 

def split_into_chunks(doc_file: str) -> list:
    content = ""
    # Use PyMuPDF open the PDF file
    with fitz.open(doc_file) as doc:
        for page in doc:
            content += page.get_text() + "\n\n"
    
    # Maintain my original logic: Split by double newlines.
    return [chunk for chunk in content.split("\n\n") if chunk.strip()]

chunks = split_into_chunks("CECL_implementation_model_risk_WP_2024_03_version.pdf")

for i, chunk in enumerate(chunks[:5]): # print the firs 5 paragraphs
    print(f"[{i}] {chunk}\n")

C:\Users\chen_\AppData\Local\uv\cache\builds-v0\.tmpnO11e9\Scripts\python.exe: No module named pip


[0] ISSN: 1962-5361
Disclaimer: This Philadelphia Fed working paper represents preliminary research that is being 
circulated for discussion purposes. The views expressed in these papers are solely those of  
the authors and do not necessarily reflect the views of the Federal Reserve Bank of Philadelphia 
or the Federal Reserve System. Any errors or omissions are the responsibility of the authors. 
Philadelphia Fed working papers are free to download at: https://philadelphiafed.org/research-
and-data/publications/working-papers.
Working Papers
RESEARCH DEPARTMENT
DOI: https://doi.org/10.21799/frbp.wp.2024.03
José J. Canals-Cerdá
Federal Reserve Bank of Philadelphia Supervision, Regulation, and 
Credit Department
CECL Implementation 
and Model Risk in 
Uncertain Times
An Application to Consumer 
Finance
WP 24-03
PUBLISHED
February 2024

[1] 
CECL Implementation and Model Risk in Uncertain Times: An 
Application to Consumer Finance 
By José J. Canals-Cerdá1 
Abstract 
I examine the chall

### Step 2: Convert text paragraphs into large vectors

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-mpnet-base-v2")

def embed_chunk(chunk: str) -> List[float]:
    embedding = embedding_model.encode(chunk, normalize_embeddings=True)
    return embedding.tolist()


embedding = embed_chunk("Stress testing scenarios for macroeconomic shocks")
print(len(embedding))
print(embedding)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

768
[-0.053258899599313736, -0.03900706395506859, -0.05442294105887413, -0.007973871193826199, -0.0010782555909827352, 0.01798773929476738, -0.01974315010011196, 0.040681373327970505, 0.035838089883327484, 0.03330278769135475, 0.049479998648166656, 0.01739715039730072, -0.03789912909269333, -0.02896641381084919, 0.01456045638769865, -0.04352908208966255, 0.0625964105129242, -0.009620953351259232, 0.014774478040635586, -0.010203198529779911, -0.018137862905859947, 0.00796608068048954, -0.0026146185118705034, 0.027915718033909798, -0.03376911208033562, -0.010286042466759682, 0.01857621595263481, 0.030126260593533516, 0.019850116223096848, -0.014607593417167664, -0.028103450313210487, 0.01648547686636448, 0.003361326642334461, -0.011269662529230118, 1.2036119869662798e-06, -0.0027217036113142967, -0.03554821386933327, -0.05349332094192505, 0.005267336033284664, 0.028614317998290062, -0.001479694852605462, -0.04149001091718674, -0.002162008313462138, 0.024716150015592575, -0.02996634691953

In [12]:
embeddings = [embed_chunk(chunk) for chunk in chunks]

print(len(embeddings))
print(embeddings[0])

47
[-0.007001870311796665, 0.012467384338378906, -0.04730839282274246, -0.0731939896941185, 0.02429821342229843, 0.07826381176710129, -0.021717021241784096, 0.028593959286808968, -0.03276882320642471, 0.024380208924412727, 0.021572312340140343, 0.021872062236070633, 0.007044448051601648, 0.03137633576989174, -0.01381775364279747, 0.021792879328131676, 0.011588902212679386, 0.003379395231604576, 0.0832371637225151, 0.003921246621757746, 0.034656040370464325, -0.06652142852544785, 0.008830235339701176, 0.0070910220965743065, -0.03557242825627327, -0.048490483313798904, 0.025512991473078728, -0.020256804302334785, -0.03147665038704872, 0.004751345608383417, 0.030313728377223015, -0.013153216801583767, 0.0069540333934128284, 0.016285773366689682, 2.522604972909903e-06, -0.043332040309906006, -0.03072691336274147, -0.0395156554877758, -0.017397785559296608, -0.001435255166143179, -0.024947483092546463, -0.005538827273994684, 0.024936839938163757, -0.01887236163020134, -0.05173404887318611, 

### Step 3: Vector Database

In [13]:
# The following code block is the bridge between my raw CECL text and a searchable "AI memory." 
# It initializes a Vector Database (ChromaDB) and populates it with my processed CECL data.
#
# Here is the step-by-step breakdown of the logic:
#   1. Initializing the Vector Databasechromadb.EphemeralClient(): 
#      This creates an in-memory database. 
#      It’s "ephemeral," meaning the data lives in my RAM and will be deleted once I stop the Python script. 
#      This is perfect for rapid prototyping or running a demo.get_or_create_collection(name="default"): 
#       A "Collection" in ChromaDB is like a Table in a traditional SQL database. It organizes my data under 
#       a specific name.
#   2. The save_embeddings Function
#      This function takes my text chunks and their corresponding mathematical 
#      "fingerprints" (embeddings) and stores them so they can be searched later.zip(chunks, embeddings): It pairs 
#      each text string with its specific vector.


In [14]:
import chromadb

chromadb_client = chromadb.EphemeralClient()
chromadb_collection = chromadb_client.get_or_create_collection(name="default")

def save_embeddings(chunks: List[str], embeddings: List[List[float]]) -> None:
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        chromadb_collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

save_embeddings(chunks, embeddings)

#### Step 3-B: "Retrieval" phase of a RAG (Retrieval-Augmented Generation) pipeline.


In [15]:
# It acts as a semantic search engine that finds the most relevant parts of my CECL document based on the meaning of 
# my question rather than just keyword matching.

In [16]:
def retrieve(query: str, top_k: int) -> List[str]:
    query_embedding = embed_chunk(query)
    results = chromadb_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['documents'][0]

query = "what is the key summary of CECL stress test？"
retrieved_chunks = retrieve(query, 10)

for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i}] {chunk}\n")

[0] 
44 
 
A. 
APPENDIX: Regulatory Guidance on CECL Implementation. 
 
As FASB staff has indicated in multiple instances, the CECL standard allows for flexibility in 
determining the best approach for computing the allowance. CECL is by design nonprescriptive 
about the methodology that should be employed when computing the allowance, as well as the 
economic projections that should be considered when determining the reasonable and 
supportable forecast. This level of flexibility is intended to facilitate CECL implementation across 
financial institutions with different levels of complexity.  
For the less sophisticated financial institutions, banking regulators have contributed examples of 
acceptable methodologies, like the snapshot/open pool approach, the vintage approach, and the 
remaining life/weighted average remaining maturity (WARM) approach.34 The methods differ 
primarily on the way the lifetime historical charge-off rate is calculated. For example, the 
snapshot approach c

### Step 4: Use CrossEncoder to sort and select the top selected paragraphs / answers

In [17]:
from sentence_transformers import CrossEncoder

def rerank(query: str, retrieved_chunks: List[str], top_k: int) -> List[str]:
    cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)

    scored_chunks = list(zip(retrieved_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return [chunk for chunk, _ in scored_chunks][:top_k]

reranked_chunks = rerank(query, retrieved_chunks, 5)

for i, chunk in enumerate(reranked_chunks):
    print(f"[{i}] {chunk}\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[0] 
3 
 
The next section introduces the CECL framework in greater detail and provides a brief overview 
of the relevant literature. Section three analyzes the initial impact of CECL implementation on 
the allowances of financial institutions as well as the differential impact of the pandemic on the 
allowances across CECL adopters and nonadopters. Section four analyzes conceptually the impact 
of economic forecasting error and model misspecification error on CECL allowances. Section five 
introduces a simple empirical framework for CECL implementation with an application for auto 
loans as a particular example of a consumer finance portfolio. Section six discusses empirical 
findings and lessons learned on how to mitigate potential CECL projection bias in times of high 
economic uncertainty. Section seven concludes. An appendix provides some additional 
background on regulatory guidance regarding CECL implementation. 
II. 
The CECL Framework: A Brief Introduction 
ALLL is an estimate

### Step 5: "Generation" stage

In [18]:
# "Generation" stageis the final and most critical component of the RAG (Retrieval-Augmented Generation) pipeline. 
# It takes the raw data I retrieved earlier and uses a Large Language Model (LLM) to transform it into a coherent, human-readable answer.

In [19]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
google_client = genai.Client()

def generate(query: str, chunks: List[str]) -> str:
    prompt = f"""You are a knowledge assistant. Please generate accurate responses based on the user's questions and the following snippets.

User Question: {query}

Relevant context:
{"\n\n".join(chunks)}

Please answer based on the content above; do not hallucinate or fabricate information."""

    print(f"{prompt}\n\n************************* The following answer is from RAG's conclusion *************************\n")

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

answer = generate(query, reranked_chunks)
print(answer)

You are a knowledge assistant. Please generate accurate responses based on the user's questions and the following snippets.

User Question: what is the key summary of CECL stress test？

Relevant context:

3 
 
The next section introduces the CECL framework in greater detail and provides a brief overview 
of the relevant literature. Section three analyzes the initial impact of CECL implementation on 
the allowances of financial institutions as well as the differential impact of the pandemic on the 
allowances across CECL adopters and nonadopters. Section four analyzes conceptually the impact 
of economic forecasting error and model misspecification error on CECL allowances. Section five 
introduces a simple empirical framework for CECL implementation with an application for auto 
loans as a particular example of a consumer finance portfolio. Section six discusses empirical 
findings and lessons learned on how to mitigate potential CECL projection bias in times of high 
economic uncertai

In [20]:

from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-04-23 00:36:55.506106-04:00
Finished: 2026-04-23 00:37:31.326480-04:00

Duration: 0:00:35.820374
Duration: 35.820 seconds
